In [2]:
import torch

import torch
import torch.nn as nn

class MultiHeadAttention(nn.Module):
    """
    Multi-head causal self-attention.

    Splits the embedding dimension across multiple heads, computes scaled
    dot-product attention independently per head (with causal masking so
    each token only attends to itself and earlier tokens), then
    concatenates the head outputs and projects back to the embedding dim.

    Args:
        d_in: input embedding dimension
        d_out: output embedding dimension (must be divisible by n_heads)
        context_length: max sequence length, used to size the causal mask
        dropout: dropout probability applied to attention weights
        n_heads: number of attention heads
        qkv_bias: whether the Q/K/V linear projections use a bias term
    """
    def __init__(
            self, 
            d_in: int,
            d_out: int,
            context_length: int,
            dropout: float,
            n_heads: int,
            qkv_bias: bool,
    ):
        super().__init__()
        assert d_out % n_heads == 0, "d_out must be divisible by n_heads"

        self.d_out = d_out
        self.n_heads = n_heads
        self.head_dim = d_out // n_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)

        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: input tensor of shape (batch_size, num_tokens, d_in)

        Returns:
            Tensor of shape (batch_size, num_tokens, d_out)
        """
        batch_size, num_tokens, _ = x.shape

        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        # Split d_out into (n_heads, head_dim), then move heads before tokens
        queries = queries.view(batch_size, num_tokens, self.n_heads, self.head_dim).transpose(1, 2)
        keys = keys.view(batch_size, num_tokens, self.n_heads, self.head_dim).transpose(1, 2)
        values = values.view(batch_size, num_tokens, self.n_heads, self.head_dim).transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)

        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1, 2)
        context_vec = context_vec.contiguous().view(batch_size, num_tokens, self.d_out)

        return self.out_proj(context_vec)


# --- Small dummy values for quick testing ---
batch_size = 2
num_tokens = 6
d_in = 768       # embedding dimension in
d_out = 768      # embedding dimension out (must match d_in for residual connections later)
n_heads = 12
context_length = 1024
dropout = 0.0    # set to 0 for now so outputs are deterministic while testing

torch.manual_seed(123)

# Fake input: pretend this is already-embedded token vectors
x = torch.rand(batch_size, num_tokens, d_in)
print("Input shape:", x.shape)

mha = MultiHeadAttention(
    d_in=d_in,
    d_out=d_out,
    context_length=context_length,
    dropout=dropout,
    n_heads=n_heads,
    qkv_bias=False,
)

# context_vecs = mha(x)
# print("Output shape:", context_vecs.shape)
# assert context_vecs.shape == (batch_size, num_tokens, d_out), "Shape mismatch!"
# print("✅ Shape check passed")

Input shape: torch.Size([2, 6, 768])


In [10]:
import torch
from torch import nn

batch_size = 2
num_tokens = 6
d_in = 768
d_out = 768
n_heads = 12
context_length = 1024
dropout = 0.0

x = torch.rand(batch_size, num_tokens, d_in)


In [13]:
W_query = nn.Linear(d_in, d_out, bias=False)

In [15]:
W_query(x)

tensor([[[-0.0197, -0.3935,  0.2645,  ..., -0.1249,  0.1223, -0.5718],
         [-0.2124, -0.5525,  0.3660,  ..., -0.1106,  0.1983, -0.9219],
         [-0.2715, -0.4467,  0.1148,  ..., -0.0299,  0.0984, -0.4590],
         [ 0.1105, -0.4403,  0.4411,  ..., -0.0323, -0.0179, -0.4827],
         [ 0.0665, -0.4671,  0.0130,  ..., -0.1835,  0.3409, -0.3832],
         [ 0.0382, -0.2006,  0.1046,  ..., -0.1309,  0.5059, -0.8623]],

        [[ 0.1847, -0.3517, -0.0643,  ..., -0.1457,  0.1230, -0.5029],
         [ 0.0076, -0.5040,  0.2291,  ..., -0.3510, -0.0534, -0.6505],
         [-0.1952, -0.1306,  0.0852,  ..., -0.5439,  0.2929, -0.6554],
         [-0.3267, -0.5297,  0.3391,  ..., -0.0274,  0.2182, -0.8372],
         [-0.2531, -0.4815,  0.3284,  ..., -0.5619,  0.1918, -0.3630],
         [-0.1475, -0.2866,  0.0388,  ..., -0.1928, -0.0718, -0.7953]]],
       grad_fn=<UnsafeViewBackward0>)

In [5]:
import torch
import torch.nn as nn


class MultiHeadAttentionVerbose(nn.Module):
    """Same as MultiHeadAttention, but prints shapes at each step for learning."""

    def __init__(self, d_in, d_out, context_length, dropout, n_heads, qkv_bias=False):
        super().__init__()
        assert d_out % n_heads == 0

        self.d_out = d_out
        self.n_heads = n_heads
        self.head_dim = d_out // n_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)

        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1),
        )

    def forward(self, x):
        batch_size, num_tokens, _ = x.shape
        print(f"1. Input x:              {tuple(x.shape)}")

        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)
        print(f"2. After W_q/W_k/W_v:    {tuple(queries.shape)}  (still one 768-vector per token)")

        queries = queries.view(batch_size, num_tokens, self.n_heads, self.head_dim).transpose(1, 2)
        keys = keys.view(batch_size, num_tokens, self.n_heads, self.head_dim).transpose(1, 2)
        values = values.view(batch_size, num_tokens, self.n_heads, self.head_dim).transpose(1, 2)
        print(f"3. After split+transpose:{tuple(queries.shape)}  (batch, heads, tokens, head_dim)")

        attn_scores = queries @ keys.transpose(2, 3)
        print(f"4. Raw attn_scores:      {tuple(attn_scores.shape)}  (token-to-token scores, per head)")

        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        print(f"5. After masking, row 0: {attn_scores[0, 0, 0].tolist()}")

        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        print(f"6. attn_weights row 0:   {[round(v, 3) for v in attn_weights[0, 0, 0].tolist()]}")
        print(f"   (sums to: {attn_weights[0, 0, 0].sum().item():.3f})")

        context_vec = (attn_weights @ values).transpose(1, 2)
        print(f"7. After weighted sum:   {tuple(context_vec.shape)}  (heads back next to tokens)")

        context_vec = context_vec.contiguous().view(batch_size, num_tokens, self.d_out)
        print(f"8. After merging heads:  {tuple(context_vec.shape)}  (back to one 768-vector per token)")

        output = self.out_proj(context_vec)
        print(f"9. Final output:         {tuple(output.shape)}")

        return output


# --- Run it ---
torch.manual_seed(123)
x = torch.rand(2, 6, 768)

mha_verbose = MultiHeadAttentionVerbose(
    d_in=768, d_out=768, context_length=1024, dropout=0.0, n_heads=12, qkv_bias=False,
)

_ = mha_verbose(x)

1. Input x:              (2, 6, 768)
2. After W_q/W_k/W_v:    (2, 6, 768)  (still one 768-vector per token)
3. After split+transpose:(2, 12, 6, 64)  (batch, heads, tokens, head_dim)
4. Raw attn_scores:      (2, 12, 6, 6)  (token-to-token scores, per head)
5. After masking, row 0: [0.5401889681816101, -inf, -inf, -inf, -inf, -inf]
6. attn_weights row 0:   [1.0, 0.0, 0.0, 0.0, 0.0, 0.0]
   (sums to: 1.000)
7. After weighted sum:   (2, 6, 12, 64)  (heads back next to tokens)
8. After merging heads:  (2, 6, 768)  (back to one 768-vector per token)
9. Final output:         (2, 6, 768)
